In [1]:
# Cell 1 - Imports and file paths
import os
from datetime import timedelta
import numpy as np
import pandas as pd
import networkx as nx
import json

# Files produced by 01_simulate_data.ipynb
RAW_TX_PATH = "raw_data/simulated_transactions.csv"
ACCOUNTS_PATH = "raw_data/accounts_simulated.csv"
LABELS_PATH = "raw_data/account_labels.csv"

# Output
OUT_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)
FEATURES_OUTPATH = os.path.join(OUT_DIR, "features_accounts.csv")

print("Paths set. Raw tx:", RAW_TX_PATH)


Paths set. Raw tx: raw_data/simulated_transactions.csv


In [2]:
# Cell 2 - Load CSVs and ensure timestamp dtype
tx = pd.read_csv(RAW_TX_PATH)
accounts = pd.read_csv(ACCOUNTS_PATH)
labels = pd.read_csv(LABELS_PATH)

# ensure timestamp parsed
if not np.issubdtype(tx["timestamp"].dtype, np.datetime64):
    tx["timestamp"] = pd.to_datetime(tx["timestamp"])

print("Loaded:", len(accounts), "accounts,", len(tx), "transactions")
tx.head(3)


Loaded: 5000 accounts, 27048 transactions


,tx_id,timestamp,sender,receiver,amount
0,T0011450,2025-10-20 22:43:36.608166,A003020,A004120,925.18
1,T0007410,2025-10-20 22:45:26.608166,A002906,A002865,2714.12
2,T0016049,2025-10-20 22:47:12.608166,A003892,A001515,703.90


In [3]:
# Cell 3 - helper columns and windows we'll use
tx = tx.sort_values("timestamp").reset_index(drop=True)

# windows (relative to the last timestamp in dataset)
last_ts = tx["timestamp"].max()
window_24h_start = last_ts - timedelta(days=1)
window_7d_start = last_ts - timedelta(days=7)

print("Last timestamp:", last_ts)
print("24h window starts at:", window_24h_start)
print("7d window starts at:", window_7d_start)


Last timestamp: 2025-11-19 22:42:40.608166
24h window starts at: 2025-11-18 22:42:40.608166
7d window starts at: 2025-11-12 22:42:40.608166


In [4]:
# Cell 4 - Compute behavioral features for each account
# We'll compute for receiver (inbound) and sender (outbound) separately, in 24h and 7d windows.

def agg_counts(df, group_col, time_mask, prefix):
    sub = df[time_mask].groupby(group_col).agg(
        tx_count = ("tx_id","count"),
        tx_sum = ("amount","sum"),
        unique_counterparts = (group_col, lambda s: 0)  # placeholder overwritten below
    ).rename(columns={"tx_count": f"{prefix}_count", "tx_sum": f"{prefix}_sum"})
    return sub

# 24h inbound: count of transactions where account is receiver in last 24h
mask_24_in = tx["timestamp"] >= window_24h_start
in_24 = tx[mask_24_in].groupby("receiver").agg(
    inbound_count_24h = ("tx_id","count"),
    inbound_sum_24h = ("amount","sum"),
    inbound_unique_senders_24h = ("sender", lambda s: s.nunique())
).reset_index().rename(columns={"receiver":"account_id"})

# 7d inbound:
mask_7_in = tx["timestamp"] >= window_7d_start
in_7 = tx[mask_7_in].groupby("receiver").agg(
    inbound_count_7d = ("tx_id","count"),
    inbound_sum_7d = ("amount","sum"),
    inbound_unique_senders_7d = ("sender", lambda s: s.nunique())
).reset_index().rename(columns={"receiver":"account_id"})

# 24h outbound:
out_24 = tx[mask_24_in].groupby("sender").agg(
    outbound_count_24h = ("tx_id","count"),
    outbound_sum_24h = ("amount","sum"),
    outbound_unique_receivers_24h = ("receiver", lambda s: s.nunique())
).reset_index().rename(columns={"sender":"account_id"})

# 7d outbound:
out_7 = tx[mask_7_in].groupby("sender").agg(
    outbound_count_7d = ("tx_id","count"),
    outbound_sum_7d = ("amount","sum"),
    outbound_unique_receivers_7d = ("receiver", lambda s: s.nunique())
).reset_index().rename(columns={"sender":"account_id"})

# Merge behavioral features into a single frame keyed by account_id
from functools import reduce
dfs = [in_24, in_7, out_24, out_7]
features = reduce(lambda left,right: pd.merge(left,right,on="account_id",how="outer"), dfs)
features = features.fillna(0)  # missing => zero activity
features.head(5)


,account_id,inbound_count_24h,inbound_sum_24h,inbound_unique_senders_24h,inbound_count_7d,inbound_sum_7d,inbound_unique_senders_7d,outbound_count_24h,outbound_sum_24h,outbound_unique_receivers_24h,outbound_count_7d,outbound_sum_7d,outbound_unique_receivers_7d
0,A000000,0.0,0.0,0.0,1.0,2083.89,1.0,0.0,0.00,0.0,0.0,0.00,0.0
1,A000001,0.0,0.0,0.0,1.0,705.77,1.0,0.0,0.00,0.0,2.0,3188.86,2.0
2,A000002,0.0,0.0,0.0,1.0,70.66,1.0,0.0,0.00,0.0,2.0,6893.80,2.0
3,A000004,0.0,0.0,0.0,2.0,1199.42,2.0,0.0,0.00,0.0,0.0,0.00,0.0
4,A000005,0.0,0.0,0.0,1.0,1004.39,1.0,1.0,2747.67,1.0,1.0,2747.67,1.0


In [6]:
# Cell 5 (fixed) - Forwarding ratio and average forward delay (safe handling of missing lists)
epsilon = 1e-9

# Ensure timestamps are datetimes (defensive)
if not np.issubdtype(tx["timestamp"].dtype, np.datetime64):
    tx["timestamp"] = pd.to_datetime(tx["timestamp"])

# We'll compute delays using the last 7 days for speed
recent_tx = tx[tx["timestamp"] >= window_7d_start]

# Prepare per-account lists of incoming and outgoing timestamps
incoming = recent_tx.groupby("receiver").agg(in_times = ("timestamp", lambda s: list(s))).reset_index().rename(columns={"receiver":"account_id"})
outgoing = recent_tx.groupby("sender").agg(out_times = ("timestamp", lambda s: list(s))).reset_index().rename(columns={"sender":"account_id"})

# Merge and ensure missing lists are actual empty lists (can't use fillna with list)
io = pd.merge(incoming, outgoing, on="account_id", how="outer")

# Replace NaN in list-columns with empty lists safely
io["in_times"] = io["in_times"].apply(lambda x: x if isinstance(x, list) else [])
io["out_times"] = io["out_times"].apply(lambda x: x if isinstance(x, list) else [])

def avg_forward_delay_seconds(row):
    in_times = sorted(row["in_times"])
    out_times = sorted(row["out_times"])
    if len(in_times) == 0 or len(out_times) == 0:
        return np.nan
    delays = []
    j = 0
    # For each inbound time find the first outgoing time strictly after it
    for t_in in in_times:
        while j < len(out_times) and out_times[j] <= t_in:
            j += 1
        if j < len(out_times):
            delta = (out_times[j] - t_in).total_seconds()
            if delta >= 0:
                delays.append(delta)
    if len(delays) == 0:
        return np.nan
    return float(np.mean(delays))

# Compute average forward delay per account (seconds)
io["avg_forward_delay_seconds"] = io.apply(avg_forward_delay_seconds, axis=1)

# Merge avg delay into features (features must already have been merged with labels earlier in your notebook)
# If features not yet defined, ensure features exists by merging in_24/in_7/out_24/out_7 first
# Here we assume `features` exists; if not, run the earlier aggregation cells first.
features = features.merge(io[["account_id", "avg_forward_delay_seconds"]], on="account_id", how="left")

# Compute forwarding ratio using 7d sums (out / (in + eps))
features["pct_forwarded_7d"] = features["outbound_sum_7d"] / (features["inbound_sum_7d"] + epsilon)

# Fill NaNs from avg_forward_delay with sentinel (so model sees 'no forwards' distinctly)
features["avg_forward_delay_seconds"] = features["avg_forward_delay_seconds"].fillna(-1.0)

# Show a quick preview for verification
features[["account_id","inbound_count_24h","outbound_count_24h","pct_forwarded_7d","avg_forward_delay_seconds"]].head(8)


,account_id,inbound_count_24h,outbound_count_24h,pct_forwarded_7d,avg_forward_delay_seconds
0,A000000,0.0,0.0,0.000000,-1.0
1,A000001,0.0,0.0,4.518271,85481.0
2,A000002,0.0,0.0,97.562978,-1.0
3,A000004,0.0,0.0,0.000000,-1.0
4,A000005,0.0,1.0,2.735660,460746.0
5,A000006,0.0,0.0,16.944106,272674.5
6,A000007,0.0,0.0,0.000000,-1.0
7,A000009,0.0,0.0,0.000000,-1.0


In [7]:
# Cell 6 - Build a directed graph from the last 7 days and compute in/out degree and pagerank
G = nx.DiGraph()
recent = tx[tx["timestamp"] >= window_7d_start]

# add edges (sender -> receiver); include weight by count
edge_counts = recent.groupby(["sender","receiver"]).size().reset_index(name="weight")

for _, r in edge_counts.iterrows():
    G.add_edge(r["sender"], r["receiver"], weight=int(r["weight"]))

# Compute degrees (fast)
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())

# PageRank (uses weights if available)
try:
    pagerank = nx.pagerank(G, weight="weight")
except Exception:
    pagerank = {n: 0.0 for n in G.nodes()}

# create dataframe
graph_feats = pd.DataFrame({
    "account_id": list(set(list(in_deg.keys()) + list(out_deg.keys()))),
})
graph_feats["in_degree_7d"] = graph_feats["account_id"].map(in_deg).fillna(0).astype(int)
graph_feats["out_degree_7d"] = graph_feats["account_id"].map(out_deg).fillna(0).astype(int)
graph_feats["pagerank_7d"] = graph_feats["account_id"].map(pagerank).fillna(0.0).astype(float)

# merge with main features
features = features.merge(graph_feats, on="account_id", how="left")
features[["account_id","in_degree_7d","out_degree_7d","pagerank_7d"]].fillna(0).head()


,account_id,in_degree_7d,out_degree_7d,pagerank_7d
0,A000000,1,0,0.000214
1,A000001,1,2,0.000177
2,A000002,1,2,0.000242
3,A000004,2,0,0.000223
4,A000005,1,1,0.000118


In [8]:
# Cell 7 - Add some derived features the model likes & normalize-ish columns
# 1) ratio inbound/outbound counts (7d)
features["count_in_out_ratio_7d"] = (features["inbound_count_7d"] + epsilon) / (features["outbound_count_7d"] + epsilon)

# 2) flag: many unique incoming senders in 24h (heuristic)
features["many_inbound_senders_24h_flag"] = (features["inbound_unique_senders_24h"] >= 10).astype(int)

# 3) burstiness proxy: inbound_count_24h relatively large compared to 7d average
features["burstiness_24h_vs_7d"] = features["inbound_count_24h"] / ( (features["inbound_count_7d"]/7.0) + epsilon )

# Fill any remaining NaNs with zeros (except is_mule preserve)
cols_to_fill = [c for c in features.columns if c not in ("account_id","is_mule")]
features[cols_to_fill] = features[cols_to_fill].fillna(0)
features.head(4)


,account_id,inbound_count_24h,inbound_sum_24h,inbound_unique_senders_24h,inbound_count_7d,inbound_sum_7d,inbound_unique_senders_7d,outbound_count_24h,outbound_sum_24h,outbound_unique_receivers_24h,...,outbound_sum_7d,outbound_unique_receivers_7d,avg_forward_delay_seconds,pct_forwarded_7d,in_degree_7d,out_degree_7d,pagerank_7d,count_in_out_ratio_7d,many_inbound_senders_24h_flag,burstiness_24h_vs_7d
0,A000000,0.0,0.0,0.0,1.0,2083.89,1.0,0.0,0.0,0.0,...,0.00,0.0,-1.0,0.000000,1,0,0.000214,1.000000e+09,0,0.0
1,A000001,0.0,0.0,0.0,1.0,705.77,1.0,0.0,0.0,0.0,...,3188.86,2.0,85481.0,4.518271,1,2,0.000177,5.000000e-01,0,0.0
2,A000002,0.0,0.0,0.0,1.0,70.66,1.0,0.0,0.0,0.0,...,6893.80,2.0,-1.0,97.562978,1,2,0.000242,5.000000e-01,0,0.0
3,A000004,0.0,0.0,0.0,2.0,1199.42,2.0,0.0,0.0,0.0,...,0.00,0.0,-1.0,0.000000,2,0,0.000223,2.000000e+09,0,0.0


In [9]:
# Cell 8 - Select final feature set, attach label, and save
final_cols = [
    "account_id", "is_mule",
    "inbound_count_24h","inbound_sum_24h","inbound_unique_senders_24h",
    "inbound_count_7d","inbound_sum_7d","inbound_unique_senders_7d",
    "outbound_count_24h","outbound_sum_24h","outbound_unique_receivers_24h",
    "outbound_count_7d","outbound_sum_7d","outbound_unique_receivers_7d",
    "pct_forwarded_7d","avg_forward_delay_seconds",
    "in_degree_7d","out_degree_7d","pagerank_7d",
    "count_in_out_ratio_7d","many_inbound_senders_24h_flag","burstiness_24h_vs_7d"
]

# Ensure all columns exist
for c in final_cols:
    if c not in features.columns:
        features[c] = 0

features_final = features[final_cols].copy()
features_final.to_csv(FEATURES_OUTPATH, index=False)
print("Saved feature table to:", FEATURES_OUTPATH)
features_final.head(5)


Saved feature table to: data\features_accounts.csv


,account_id,is_mule,inbound_count_24h,inbound_sum_24h,inbound_unique_senders_24h,inbound_count_7d,inbound_sum_7d,inbound_unique_senders_7d,outbound_count_24h,outbound_sum_24h,...,outbound_sum_7d,outbound_unique_receivers_7d,pct_forwarded_7d,avg_forward_delay_seconds,in_degree_7d,out_degree_7d,pagerank_7d,count_in_out_ratio_7d,many_inbound_senders_24h_flag,burstiness_24h_vs_7d
0,A000000,0,0.0,0.0,0.0,1.0,2083.89,1.0,0.0,0.00,...,0.00,0.0,0.000000,-1.0,1,0,0.000214,1.000000e+09,0,0.0
1,A000001,0,0.0,0.0,0.0,1.0,705.77,1.0,0.0,0.00,...,3188.86,2.0,4.518271,85481.0,1,2,0.000177,5.000000e-01,0,0.0
2,A000002,0,0.0,0.0,0.0,1.0,70.66,1.0,0.0,0.00,...,6893.80,2.0,97.562978,-1.0,1,2,0.000242,5.000000e-01,0,0.0
3,A000004,0,0.0,0.0,0.0,2.0,1199.42,2.0,0.0,0.00,...,0.00,0.0,0.000000,-1.0,2,0,0.000223,2.000000e+09,0,0.0
4,A000005,0,0.0,0.0,0.0,1.0,1004.39,1.0,1.0,2747.67,...,2747.67,1.0,2.735660,460746.0,1,1,0.000118,1.000000e+00,0,0.0


In [10]:
# Cell 9 - quick checks to understand label balance and feature ranges
print("Shape:", features_final.shape)
print("Label distribution:")
print(features_final["is_mule"].value_counts(normalize=False))

# Show summary stat for some key columns
print("\nSummary stats (selected):")
print(features_final[["inbound_count_7d","outbound_count_7d","pct_forwarded_7d","in_degree_7d","pagerank_7d"]].describe().T)


Shape: (4387, 22)
Label distribution:
is_mule
0    4387
Name: count, dtype: int64

Summary stats (selected):
                    count          mean           std       min       25%  \
inbound_count_7d   4387.0  1.401413e+00  2.674980e+00  0.000000  0.000000   
outbound_count_7d  4387.0  1.401413e+00  1.766339e+00  0.000000  1.000000   
pct_forwarded_7d   4387.0  7.381523e+11  1.769389e+12  0.000000  0.049881   
in_degree_7d       4387.0  1.400501e+00  2.672644e+00  0.000000  0.000000   
pagerank_7d        4387.0  2.279462e-04  3.171232e-04  0.000083  0.000083   

                        50%           75%           max  
inbound_count_7d   1.000000  2.000000e+00  5.100000e+01  
outbound_count_7d  1.000000  2.000000e+00  3.500000e+01  
pct_forwarded_7d   1.302258  3.677400e+11  2.352322e+13  
in_degree_7d       1.000000  2.000000e+00  5.100000e+01  
pagerank_7d        0.000148  2.403263e-04  4.754997e-03  
